In [1]:
#!/usr/bin/env python3
import os
import csv
import wave
import whisper
import librosa
import numpy as np
from pydub import AudioSegment
from pathlib import Path

# Configuration
MODEL_SIZE = "small.en"          # Whisper model size
CHUNK_DURATION = 30               # Seconds per chunk
AUDIO_DIR = Path("./raw_cumberbatch_data")       # Input audio directory
CHUNKS_DIR = Path("./dataset/chunks")     # Output directory for chunks
CSV_PATH = Path("./dataset/transcriptions.csv")  # Output CSV file
STATE_FILE = Path("./dataset/processing_state.txt")  # Resume state tracking

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)

def load_resume_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, 'r') as f:
            return int(f.read().strip())
    return 0

def save_resume_state(last_index):
    with open(STATE_FILE, 'w') as f:
        f.write(str(last_index))

def convert_to_16k_mono(input_path, output_path):
    """Convert audio file to 16kHz mono WAV"""
    try:
        audio = AudioSegment.from_file(input_path)
        audio = audio.set_frame_rate(16000).set_channels(1)
        audio.export(output_path, format="wav")
        return True
    except Exception as e:
        print(f"Error converting {input_path}: {str(e)}")
        return False

def split_audio(file_path, chunk_index):
    """Split audio into 30-second chunks, return chunk paths"""
    try:
        y, sr = librosa.load(file_path, sr=16000, mono=True)
        chunk_size = CHUNK_DURATION * sr
        chunks = []
        
        for i in range(0, len(y), chunk_size):
            chunk = y[i:i+chunk_size]
            chunk_path = CHUNKS_DIR / f"{chunk_index:03d}.wav"
            
            with wave.open(str(chunk_path), 'wb') as wf:
                wf.setnchannels(1)
                wf.setsampwidth(2)
                wf.setframerate(16000)
                wf.writeframes((chunk * 32767).astype(np.int16))
            
            chunks.append(chunk_path)
            chunk_index += 1
        
        return chunks, chunk_index
    except Exception as e:
        print(f"Error splitting {file_path}: {str(e)}")
        return [], chunk_index

def transcribe_chunk(model, audio_path):
    """Transcribe a single audio chunk"""
    audio = whisper.load_audio(str(audio_path))
    result = model.transcribe(audio, fp16=False, verbose=False)
    return result["text"].strip()

def main():
    # Create directories if needed
    ensure_dir(AUDIO_DIR)
    ensure_dir(CHUNKS_DIR)
    
    # Initialize Whisper model
    model = whisper.load_model(MODEL_SIZE)
    
    # Resume state management
    chunk_index = load_resume_state()
    processed_files = set()
    
    # Load existing transcriptions if resuming
    if CSV_PATH.exists():
        with open(CSV_PATH, 'r') as f:
            reader = csv.reader(f)
            next(reader, None)  # Skip header
            for row in reader:
                if row: processed_files.add(row[0])
    
    # Process audio files in directory
    audio_files = sorted(AUDIO_DIR.glob("*.wav"))
    
    # Process each audio file
    with open(CSV_PATH, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        
        # Write header if new file
        if csvfile.tell() == 0:
            writer.writerow(["file_name", "transcription"])
        
        for audio_file in audio_files:
            print(f"Processing {audio_file.name}...")
            
            # Convert to proper format if needed
            converted_path = CHUNKS_DIR / "temp.wav"
            if not convert_to_16k_mono(audio_file, converted_path):
                continue
            
            # Split into chunks
            chunks, new_index = split_audio(converted_path, chunk_index)
            os.remove(converted_path)  # Cleanup temp file
            
            # Process each chunk
            for chunk_path in chunks:
                chunk_name = chunk_path.name
                
                # Skip already processed chunks
                if chunk_name in processed_files:
                    chunk_index += 1
                    continue
                
                # Transcribe and save
                try:
                    transcription = transcribe_chunk(model, chunk_path)
                    writer.writerow([chunk_name, transcription])
                    csvfile.flush()  # Ensure immediate write
                    processed_files.add(chunk_name)
                except Exception as e:
                    print(f"Error transcribing {chunk_name}: {str(e)}")
                    # Save state before exiting on error
                    save_resume_state(chunk_index)
                    return
                
                # Update state
                chunk_index += 1
                save_resume_state(chunk_index)
    
    # Cleanup state file after successful completion
    if STATE_FILE.exists():
        os.remove(STATE_FILE)
    
    print(f"\nProcessing complete! Results saved to {CSV_PATH}")

if __name__ == "__main__":
    main()

Processing Audiobook - Benedict Cumberbatch read Casanova [oNhyLKUjRec].wav...
Transcribing 000.wav...


100%|██████████| 3000/3000 [00:02<00:00, 1186.74frames/s]


Transcribing 001.wav...


100%|██████████| 3000/3000 [00:02<00:00, 1194.03frames/s]


Transcribing 002.wav...


...

Transcribing 043.wav...


100%|██████████| 3000/3000 [00:02<00:00, 1139.12frames/s]


Transcribing 044.wav...


  0%|          | 0/3000 [00:01<?, ?frames/s]


KeyboardInterrupt: 